**Feature Selection Method:** FLAML   
**Dataset:** QUT-DV25 (TCP)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif
from flaml import AutoML

In [ ]:

# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [ ]:
DATASET_NAME = "TCP"
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [ ]:
gc.collect()
tf.keras.backend.clear_session()


In [ ]:
data.head()

,Package_Name,Total_Entries,Unique_C-COMM,Python_Related_Process,State_Transition,Local_IP_Address_Access,Remote_IP_Address_Access,Local_Port_Access,Remote_Port_Access,Level
0,10Cent10-999.0.4.tar.gz,946,7,316,"{'FIN_WAIT1 -> ->': 188, 'SYN_SENT -> ->': 187...",2,62,194,21,1
1,10Cent11-999.0.4.tar.gz,874,6,301,"{'SYN_SENT -> ->': 177, 'FIN_WAIT1 -> ->': 170...",2,51,181,18,1
2,11Cent-999.0.0.tar.gz,801,4,272,"{'FIN_WAIT1 -> ->': 160, 'SYN_SENT -> ->': 157...",2,52,164,17,1
3,11Cent-999.0.1.tar.gz,728,4,254,"{'SYN_SENT -> ->': 149, 'FIN_WAIT1 -> ->': 141...",2,39,152,14,1
4,11Cent-999.0.2.tar.gz,655,4,234,"{'SYN_SENT -> ->': 133, 'FIN_WAIT1 -> ->': 128...",2,32,135,11,1


In [ ]:
if 'Package_Name' in data.columns:
    data = data.drop(columns=['Package_Name'])

# Combine categorical columns into a single text feature
categorical_columns = [col for col in data.columns if data[col].dtype == 'object' and col != 'Level']
data['Combined_Categorical'] = data[categorical_columns].fillna('').astype(str).agg(' '.join, axis=1)

# Vectorize the combined categorical text column using n-grams
vectorizer = CountVectorizer(ngram_range=(2, 3))
categorical_ngrams = vectorizer.fit_transform(data['Combined_Categorical'])

# Ensure only valid numeric columns are selected
numerical_columns = [
    col for col in data.columns
    if pd.to_numeric(data[col], errors='coerce').notnull().all() and col != 'Level'
]

# Prepare numerical features
numerical_features = data[numerical_columns].fillna(0).astype(float)

# Convert numerical features to sparse matrix
numerical_features_sparse = csr_matrix(numerical_features.values)

# Combine numerical features with n-gram features
X = hstack([numerical_features_sparse, categorical_ngrams])

# Scale features before combining
scaler = StandardScaler(with_mean=False)  # Sparse matrices need `with_mean=False`
X_scaled = scaler.fit_transform(X)

In [ ]:
import pandas as pd
from flaml import AutoML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Assume 'data' is your pandas DataFrame
X = data.drop(columns=['Level'])
y = data['Level']

if y.dtype == 'object' or y.dtype.name == 'category':
    y = LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize FLAML
automl = AutoML()
automl_settings = {
    "time_budget": 300,
    "metric": "accuracy",
    "task": "classification",
}

automl.fit(X_train=X_train, y_train=y_train, **automl_settings)

# Feature importance & selection
selected_features = X.columns.tolist()
dropped_features = []

try:
    importances = automl.model.feature_importances_

    # Sometimes LGBM may drop unused features -> align with all columns
    if len(importances) < X.shape[1]:
        full_importances = np.zeros(X.shape[1])
        # Use feature names from model if available
        try:
            model_features = automl.model.booster_.feature_name()
            for i, col in enumerate(X.columns):
                if col in model_features:
                    idx = model_features.index(col)
                    full_importances[i] = importances[idx]
        except:
            full_importances[:len(importances)] = importances
        importances = full_importances

    varimp = pd.DataFrame({'variable': X.columns, 'importance': importances})
    varimp['relative_importance'] = varimp['importance'] / varimp['importance'].sum()

    threshold = 0.03
    selected_features = list(varimp[varimp['relative_importance'] > threshold]['variable'])
    dropped_features = list(varimp[varimp['relative_importance'] <= threshold]['variable'])

    print("Feature importance table:\n", varimp)

except AttributeError:
    print("Feature importance not available for the chosen model.")

print("\nSelected features:", selected_features)
print("Number of features kept:", len(selected_features))
print("\nDropped features:", dropped_features)
print("Number of features dropped:", len(dropped_features))


[flaml.automl.logger: 08-29 20:57:39] {1680} INFO - task = classification
[flaml.automl.logger: 08-29 20:57:39] {1691} INFO - Evaluation method: cv
[flaml.automl.logger: 08-29 20:57:39] {1789} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 08-29 20:57:39] {1901} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'lrl1']
[flaml.automl.logger: 08-29 20:57:39] {2219} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 08-29 20:57:40] {2346} INFO - Estimated sufficient time budget=3226s. Estimated necessary time budget=74s.
[flaml.automl.logger: 08-29 20:57:40] {2398} INFO -  at 0.4s,	estimator lgbm's best error=0.2597,	best estimator lgbm's best error=0.2597
[flaml.automl.logger: 08-29 20:57:40] {2219} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 08-29 20:57:40] {2398} INFO -  at 0.7s,	estimator lgbm's best error=0.2597,	best estimator lgbm's best error=0.2597
[flaml.automl.logger: 08-29 20:57:

[flaml.automl.logger: 08-29 20:58:11] {2219} INFO - iteration 34, current learner lgbm
[flaml.automl.logger: 08-29 20:58:13] {2398} INFO -  at 34.3s,	estimator lgbm's best error=0.1161,	best estimator lgbm's best error=0.1161
[flaml.automl.logger: 08-29 20:58:13] {2219} INFO - iteration 35, current learner xgboost
[flaml.automl.logger: 08-29 20:58:14] {2398} INFO -  at 34.6s,	estimator xgboost's best error=0.1481,	best estimator lgbm's best error=0.1161
[flaml.automl.logger: 08-29 20:58:14] {2219} INFO - iteration 36, current learner extra_tree
[flaml.automl.logger: 08-29 20:58:15] {2398} INFO -  at 36.0s,	estimator extra_tree's best error=0.2161,	best estimator lgbm's best error=0.1161
[flaml.automl.logger: 08-29 20:58:15] {2219} INFO - iteration 37, current learner xgboost
[flaml.automl.logger: 08-29 20:58:15] {2398} INFO -  at 36.2s,	estimator xgboost's best error=0.1481,	best estimator lgbm's best error=0.1161
[flaml.automl.logger: 08-29 20:58:15] {2219} INFO - iteration 38, curren

[flaml.automl.logger: 08-29 21:00:23] {2219} INFO - iteration 69, current learner lgbm
[flaml.automl.logger: 08-29 21:00:34] {2398} INFO -  at 174.8s,	estimator lgbm's best error=0.1126,	best estimator lgbm's best error=0.1126
[flaml.automl.logger: 08-29 21:00:34] {2219} INFO - iteration 70, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 21:00:34] {2398} INFO -  at 175.2s,	estimator xgb_limitdepth's best error=0.1204,	best estimator lgbm's best error=0.1126
[flaml.automl.logger: 08-29 21:00:34] {2219} INFO - iteration 71, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 21:00:35] {2398} INFO -  at 176.2s,	estimator xgb_limitdepth's best error=0.1188,	best estimator lgbm's best error=0.1126
[flaml.automl.logger: 08-29 21:00:35] {2219} INFO - iteration 72, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 21:00:36] {2398} INFO -  at 176.7s,	estimator xgb_limitdepth's best error=0.1188,	best estimator lgbm's best error=0.1126
[flaml.automl.logger: 08-29 21:0

[flaml.automl.logger: 08-29 21:01:51] {2398} INFO -  at 251.8s,	estimator lgbm's best error=0.1104,	best estimator lgbm's best error=0.1104
[flaml.automl.logger: 08-29 21:01:51] {2219} INFO - iteration 104, current learner extra_tree
[flaml.automl.logger: 08-29 21:01:53] {2398} INFO -  at 254.3s,	estimator extra_tree's best error=0.1271,	best estimator lgbm's best error=0.1104
[flaml.automl.logger: 08-29 21:01:53] {2219} INFO - iteration 105, current learner rf
[flaml.automl.logger: 08-29 21:01:56] {2398} INFO -  at 256.7s,	estimator rf's best error=0.1287,	best estimator lgbm's best error=0.1104
[flaml.automl.logger: 08-29 21:01:56] {2219} INFO - iteration 106, current learner extra_tree
[flaml.automl.logger: 08-29 21:01:58] {2398} INFO -  at 258.6s,	estimator extra_tree's best error=0.1271,	best estimator lgbm's best error=0.1104
[flaml.automl.logger: 08-29 21:01:58] {2219} INFO - iteration 107, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 21:02:11] {2398} INFO -  at 27

C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,


[flaml.automl.logger: 08-29 21:02:39] {2398} INFO -  at 300.2s,	estimator lrl1's best error=0.3031,	best estimator lgbm's best error=0.1104


C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,


[flaml.automl.logger: 08-29 21:02:41] {2628} INFO - retrain lgbm for 1.3s
[flaml.automl.logger: 08-29 21:02:41] {2631} INFO - retrained model: LGBMClassifier(learning_rate=0.16320328546737475, max_bin=1023,
               min_child_samples=2, n_estimators=1, n_jobs=-1, num_leaves=164,
               reg_alpha=0.013362185788458741, reg_lambda=0.006993008315649459,
               verbose=-1)
[flaml.automl.logger: 08-29 21:02:41] {1931} INFO - fit succeeded
[flaml.automl.logger: 08-29 21:02:41] {1932} INFO - Time taken to find the best model: 251.8072681427002
Feature importance table:
                    variable  importance  relative_importance
0             Total_Entries          17             0.001098
1             Unique_C-COMM           0             0.000000
2    Python_Related_Process        2043             0.131934
3          State_Transition        2123             0.137100
4   Local_IP_Address_Access        3322             0.214530
5  Remote_IP_Address_Access         405    

In [ ]:
selected_features = ['Python_Related_Process', 'State_Transition', 'Local_IP_Address_Access', 'Local_Port_Access', 'Remote_Port_Access']